# MediaPulse — Phase 4: Exploratory Data Analysis (EDA)

**Project:** MediaPulse — Media Performance & Audience Analytics  
**Target Role:** Wipro Analyst L1 (Business Analyst / Data Analyst - Media)  
**Objective:** Uncover empirical media performance dynamics, engagement patterns, category variations, and channel retention on the cleaned YouTube India trending dataset (`cleaned_youtube_india.csv`).

---

## 7 Core Analytical Investigations:
1. **Trending Activity Over Time:** Daily snapshot counts vs unique active videos.
2. **Category Performance:** Volume, median views, and engagement rates by content genre.
3. **Channel Performance:** Top 10 channels by unique trending content and audience traction.
4. **Engagement Dynamics:** Views vs. likes and comments; log-log scatter and skewness.
5. **Time to Observed Trending:** Days elapsed between upload and trending appearance.
6. **Trending Persistence:** Retention decay curve (how many days do videos remain trending?).
7. **Publication Patterns:** Upload distributions across weekdays and UTC hours.

> **Analytical Rigor Notice:** Multi-day observations of the same video represent longitudinal persistence. Video-level KPIs are aggregated using `views.max()` to avoid double-counting cumulative view counts.

### 1. Environment Setup & Data Loading

We import data science and visualization libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`) alongside our modular `src.eda` and `src.metrics` packages.

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config, metrics, eda

# Configure display options
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Load cleaned dataset
clean_csv_path = config.DATA_PROCESSED_DIR / 'cleaned_youtube_india.csv'
df = pd.read_csv(clean_csv_path)
print(f"Cleaned Dataset Loaded: {df.shape[0]:,} observations, {df.shape[1]} columns.")
print(f"Unique Videos: {df['video_id'].nunique():,}")
print(f"Unique Channels: {df['channel_title'].nunique():,}")
print(f"Trending Date Range: {df['trending_date'].min()} to {df['trending_date'].max()}")

Matplotlib is building the font cache; this may take a moment.


Cleaned Dataset Loaded: 32,265 observations, 29 columns.
Unique Videos: 16,305
Unique Channels: 1,423
Trending Date Range: 2017-11-14 to 2018-06-14


### 2. Analysis 1 — Trending Activity Over Time

**Business Question:** How consistently does the YouTube trending feed refresh, and what proportion of daily trending slots are occupied by newly trending videos versus recurring persistent videos?

We calculate daily total observations and unique video counts using `eda.compute_daily_trending_activity()`.

In [2]:
daily_activity = eda.compute_daily_trending_activity(df)
print(f"Total Observed Days: {len(daily_activity)}")
print(f"Average Daily Active Videos: {daily_activity['unique_videos'].mean():.1f}")
print(f"Max Daily Videos: {daily_activity['unique_videos'].max()} on {daily_activity.loc[daily_activity['unique_videos'].idxmax(), 'trending_date'].strftime('%Y-%m-%d')}")
print(f"Min Daily Videos: {daily_activity['unique_videos'].min()} on {daily_activity.loc[daily_activity['unique_videos'].idxmin(), 'trending_date'].strftime('%Y-%m-%d')}")

# Plot daily activity
fig_p1 = eda.plot_trending_activity(daily_activity)
print(f"Saved Figure: {fig_p1}")

Total Observed Days: 205
Average Daily Active Videos: 157.4
Max Daily Videos: 200 on 2017-12-13
Min Daily Videos: 49 on 2018-05-14


Saved Figure: D:\MediaPulse\reports\figures\01_trending_activity_over_time.png


### 3. Analysis 2 — Content Category Performance

**Business Question:** Which genres capture the largest market share of trending slots, and do high-volume categories also generate higher viewer engagement rates?

We aggregate metrics at the unique video level (`metrics.compute_video_summary_kpis()`) taking peak views and peak likes to avoid double-counting cumulative view counts.

In [3]:
category_summary = eda.compute_category_performance_summary(df)
print("Top 8 Categories by Unique Trending Videos:")
print(category_summary[['category_title', 'unique_videos', 'share_of_videos_pct', 'median_views', 'median_engagement_rate', 'mean_days_trended']].head(8).to_string(index=False))

# Plot category comparison
fig_p2 = eda.plot_category_performance(category_summary)
print(f"Saved Figure: {fig_p2}")

Top 8 Categories by Unique Trending Videos:
  category_title  unique_videos  share_of_videos_pct  median_views  median_engagement_rate  mean_days_trended
   Entertainment           7557              46.3500   202211.0000                  0.5823             1.9100
 News & Politics           2506              15.3700   148114.5000                  0.6608             1.8400
  People & Blogs           1231               7.5500   177057.0000                  0.6288             1.8900
           Music           1214               7.4500   555774.0000                  1.9044             2.5900
          Comedy           1116               6.8400   353683.0000                  4.2236             2.5700
       Education            797               4.8900    46273.0000                  5.6205             1.4400
Film & Animation            520               3.1900   495699.0000                  1.5910             2.7300
   Howto & Style            479               2.9400   243187.0000          

Saved Figure: D:\MediaPulse\reports\figures\02_category_performance_comparison.png


> **Key Finding:** Content volume is heavily concentrated in **Entertainment (46.35%)** and **News & Politics (15.37%)**. However, audience engagement rate follows an inverse pattern: specialized niche categories like **Science & Technology (7.56% median ER)** and **Education (5.62% median ER)** generate drastically higher engagement than Entertainment (0.58% median ER).

### 4. Analysis 3 — Top Channel Performance

**Business Question:** Which content publishers generate the highest number of trending videos, and what upload strategies characterize their dominance?

We aggregate at the channel grain using `eda.compute_channel_performance_summary()`.

In [4]:
top_channels = eda.compute_channel_performance_summary(df, top_n=10)
print("Top 10 Channels by Unique Trending Videos:")
print(top_channels[['channel_title', 'unique_videos', 'total_views', 'median_views', 'mean_days_trended', 'median_engagement_rate']].to_string(index=False))

# Plot channel performance
fig_p3 = eda.plot_top_channels(top_channels)
print(f"Saved Figure: {fig_p3}")

Top 10 Channels by Unique Trending Videos:
     channel_title  unique_videos  total_views  median_views  mean_days_trended  median_engagement_rate
Study IQ education            169      9637789    53255.0000             1.1800                  5.8476
         Tarang TV            158     15823246    87353.0000             1.2600                  0.4249
    etvteluguindia            153     49921892   212610.0000             1.3300                  0.3986
         VikatanTV            153    111174882   664658.0000             1.3100                  0.6069
  Saregama TVShows            152     23068307   157936.5000             1.1700                  0.6158
 Mazhavil Manorama            138     41163047   245810.0000             1.3800                  0.5898
       RadaanMedia            136     56058172   390547.5000             1.4000                  0.5420
        zeemarathi            135     18406903    96860.0000             1.2100                  0.3186
       zeetvtelugu   

Saved Figure: D:\MediaPulse\reports\figures\03_top_channels_analysis.png


### 5. Analysis 4 — Engagement Dynamics & Correlation

**Business Question:** How strongly do viewer views translate into active engagement (likes and comments)? Are engagement distributions normal or power-law distributed?

We compute Pearson and Spearman rank correlations and visualize log-log relationships.

In [5]:
correlations = eda.compute_engagement_correlations(df)
print("Engagement Metric Correlations:")
for metric_pair, r_val in correlations.items():
    print(f"  - {metric_pair}: {r_val:.4f}")

er_series = df['engagement_rate']
print(f"\nEngagement Rate Distribution:")
print(f"  Mean:   {er_series.mean():.4f}%")
print(f"  Median: {er_series.median():.4f}%")
print(f"  P25:    {er_series.quantile(0.25):.4f}%")
print(f"  P75:    {er_series.quantile(0.75):.4f}%")
print(f"  P99:    {er_series.quantile(0.99):.4f}%")
print(f"  Max:    {er_series.max():.4f}%")

# Plot scatter and distribution
fig_p4 = eda.plot_views_vs_engagement(df)
fig_p5 = eda.plot_engagement_distribution(df)
print(f"Saved Figures: {fig_p4.name}, {fig_p5.name}")

Engagement Metric Correlations:
  - pearson_views_likes: 0.8549
  - spearman_views_likes: 0.7431
  - pearson_views_comments: 0.6994
  - spearman_views_comments: 0.6655
  - pearson_likes_comments: 0.7919
  - spearman_likes_comments: 0.8813

Engagement Rate Distribution:
  Mean:   2.4232%
  Median: 1.0527%
  P25:    0.4777%
  P75:    3.0801%
  P99:    14.3208%
  Max:    77.4282%


Saved Figures: 04_views_vs_engagement_scatter.png, 05_engagement_rate_distribution.png


### 6. Analysis 5 — Time to Observed Trending (`days_to_trend`)

**Business Question:** How quickly do videos reach trending status after publication, and does trending latency vary across categories?

We analyze the distribution of calendar days elapsed between publication and trending appearance.

In [6]:
time_summary = eda.compute_time_to_trend_summary(df)
print("Time to Observed Trending by Top Categories (Days):")
print(time_summary.head(8).to_string(index=False))

dtt = df['days_to_trend']
print(f"\nOverall Days to Trend:")
print(f"  Same-day (0 days): {(dtt == 0).sum():,} ({(dtt == 0).mean()*100:.2f}%)")
print(f"  Within 2 days (<=2): {(dtt <= 2).sum():,} ({(dtt <= 2).mean()*100:.2f}%)")
print(f"  Within 7 days (<=7): {(dtt <= 7).sum():,} ({(dtt <= 7).mean()*100:.2f}%)")

# Plot time to trend by category
fig_p6 = eda.plot_time_to_trend(df)
print(f"Saved Figure: {fig_p6}")

Time to Observed Trending by Top Categories (Days):
       category_title  median_days_to_trend  mean_days_to_trend  p25_days  p75_days  min_days  max_days
            Education                1.0000              1.3500    1.0000    2.0000         0         5
Nonprofits & Activism                1.0000              1.4800    1.0000    2.0000         0         4
                Shows                1.0000              1.5100    1.0000    2.0000         1         4
     Autos & Vehicles                2.0000              2.0700    2.0000    3.0000         0         6
     Film & Animation                2.0000              2.6500    2.0000    3.0000         0        51
               Gaming                2.0000              2.4400    2.0000    3.0000         0         7
               Comedy                2.0000              2.3300    2.0000    3.0000         0         7
        Entertainment                2.0000              2.1000    2.0000    3.0000         0       221

Overall Day

Saved Figure: D:\MediaPulse\reports\figures\06_time_to_trend_by_category.png


### 7. Analysis 6 — Trending Persistence Decay

**Business Question:** How long do trending videos maintain their position on the trending list before decaying?

We count the distinct trending dates per video (`trending_date.nunique()`).

In [7]:
persistence_df = eda.compute_trending_persistence_summary(df)
print("Trending Persistence Decay Distribution:")
print(persistence_df.to_string(index=False))

# Plot persistence distribution
fig_p7 = eda.plot_trending_persistence(df)
print(f"Saved Figure: {fig_p7}")

Trending Persistence Decay Distribution:
persistence_bracket  video_count  percentage
              1 Day         7824     47.9900
             2 Days         4168     25.5600
           3-5 Days         4087     25.0700
          6-10 Days          226      1.3900
           11+ Days            0      0.0000


Saved Figure: D:\MediaPulse\reports\figures\07_trending_persistence_distribution.png


### 8. Analysis 7 — Publication Timing Patterns

**Business Question:** When are trending videos most frequently published, and how should publication hour data be interpreted?

> **Analytical Rigor Notice:** Publish hours are stored in UTC. In Indian Standard Time (IST), add 5 hours 30 minutes (e.g. 14:00 UTC = 19:30 IST). Furthermore, correlation does not imply causation: popular creators schedule uploads during evening viewing windows, creating volume concentration.

In [8]:
weekday_df, hour_df = eda.compute_publication_patterns(df)
print("Publication Volume by Weekday:")
print(weekday_df.to_string(index=False))

print("\nTop 5 Publication Hours (UTC):")
print(hour_df.sort_values(by='unique_videos', ascending=False).head(5).to_string(index=False))

# Plot publication patterns
fig_p8 = eda.plot_publication_patterns(weekday_df, hour_df)
print(f"Saved Figure: {fig_p8}")

Publication Volume by Weekday:
  weekday  unique_videos
   Monday           2246
  Tuesday           2391
Wednesday           2332
 Thursday           2503
   Friday           2694
 Saturday           2421
   Sunday           1720

Top 5 Publication Hours (UTC):
 publish_hour_utc  unique_videos
               14           1117
               16           1031
               12           1027
               13            992
               15            986


Saved Figure: D:\MediaPulse\reports\figures\08_publication_patterns.png


### 9. Summary of Key Analytical Findings for Business Intelligence

1. **Volume Concentration:** Entertainment and News account for **61.72%** of all unique trending videos.
2. **Engagement Asymmetry:** Educational and Tech content achieves **5x–12x higher median engagement rates** than mainstream entertainment.
3. **Strong Linear/Rank Correlation:** Views and likes exhibit a strong Spearman rank correlation ($ho = 0.7431$).
4. **Rapid Viral Onset:** **70.45%** of trending videos appear within 48 hours of publication.
5. **High Turnover:** **47.99%** of videos trend for only a single calendar day, reflecting high feed velocity.
6. **Publishing Cadence:** Peak publication occurs on Friday and Thursday, concentrated in the 12:00–16:00 UTC window (17:30–21:30 IST prime-time).